<a href="https://colab.research.google.com/github/MohammedElata/student-outcomes-withdrawal-prediction/blob/main/notebooks/student_outcomes_withdrawal_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Student Outcomes and Withdrawal-Risk Prediction

**BSc Data Science & Analytics Final-Year Project**

This project investigates whether academic performance, engagement and demographic indicators can help predict:

- Final academic award
- Student registration or withdrawal status
- Factors associated with withdrawal risk

The analysis uses sensitive, anonymised institutional data supplied by the **University of Portsmouth**. The underlying dataset and student-level outputs are excluded from this public notebook for privacy and data-governance reasons.

The project compares a multi-output neural network with Random Forest and XGBoost models.

> **Responsible-use statement:** This project is an analytical proof of concept. Its predictions should support—not replace—professional human judgement and should not be used to make automated decisions about individual students.

## 1. Environment Setup

In [ ]:
#Part 1
import pandas as pd
import numpy as np

## 2. Confidential Data Loading

In [ ]:
from pathlib import Path

# The original University of Portsmouth dataset is confidential and is not
# distributed with this repository. Authorised users should place the Excel
# file in the Colab session or update DATA_PATH to an approved secure location.

DATA_PATH = Path("/content/2018_student_dataset.xlsx")

if not DATA_PATH.exists():
    raise FileNotFoundError(
        "Confidential dataset not found. This file is available only to "
        "authorised users and is not included in the public repository."
    )

xls = pd.ExcelFile(DATA_PATH)

sheet_names = [
    "2018 Artefact Results",
    "2018 Module Results",
    "2018 Second Sitting",
    "Student Unit Type",
    "2018 Moodle Logs",
    "2018 Demographic",
    "2018 Placement Students",
    "2016 Demographic",
    "2017 Demographic",
    "2018 Withdrawals",
    "Foundation",
    "2018 Final Awards",
    "2015-18 Placement",
    "Placement Outcomes 2017",
]

dfs = {sheet_name: xls.parse(sheet_name) for sheet_name in sheet_names}

## 3. Data Integration and Cleaning

In [ ]:
Dataset_df = xls.parse("2018 Artefact Results")

In [ ]:
Change_NA_values = ['Withdrawal Reason', 'Withdrawal Date']
Dataset_df[Change_NA_values] = Dataset_df[Change_NA_values].fillna('Not applicable')

In [ ]:
Dataset_df.dropna(subset=['ID','1st Sitting Unit Avg', '2nd Sitting Unit Avg',
                          'Final Award', 'Gender', 'BME', 'Ethnicity Detail', 'Disability',
                          'Disability Detail', 'Mature', 'Term Postcode', 'Distance from PO1'
                          ], inplace=True)

In [ ]:
Dataset_df.drop(columns=['WP Flag', 'HESA Qual. On Entry','HESA Short Code', 'Tariff on Entry' ], inplace=True)

In [ ]:
ModuleR_df = xls.parse("2018 Module Results")

In [ ]:
ModuleR_df.dropna()

In [ ]:
withdraw_df = xls.parse("2018 Withdrawals")

In [ ]:
withdraw_df = withdraw_df.dropna()

In [ ]:
Foundation_df = xls.parse("Foundation")

In [ ]:
Foundation_df = Foundation_df.dropna()

In [ ]:
ModuleR_df["ID"]  = ModuleR_df["ID"].apply(lambda x: -1 if np.isnan(x) else int(x))

In [ ]:
Dataset_df["Pass/Fail Module?"] = ''
Dataset_df["Mark"] = ''
Dataset_df["Result"] = ''
Dataset_df["Ass Grade"] = ''
Dataset_df["2nd Sitting Mark"] = ''
Dataset_df["2nd Sitting Result"] = ''
Dataset_df["Attended Count"] = ''
Dataset_df["Not Attended"] = ''
Dataset_df["Moodle Count"] = ''
Dataset_df["Placement"] = ''
Dataset_df["HESA Qual. On Entry"] = ''
Dataset_df["HESA Short Code"] = ''

for i in Dataset_df["ID"].unique():
  filtered = ModuleR_df[ModuleR_df["ID"] == i]
  if len(filtered) > 0:
    Dataset_df.loc[Dataset_df["ID"] == i, "Pass/Fail Module?"] = filtered.iloc[0, 16]
    Dataset_df.loc[Dataset_df["ID"] == i, "Mark"] = filtered.iloc[0, 17]
    Dataset_df.loc[Dataset_df["ID"] == i, "Result"] = filtered.iloc[0, 18]
    Dataset_df.loc[Dataset_df["ID"] == i, "Ass Grade"] = filtered.iloc[0, 19]
    Dataset_df.loc[Dataset_df["ID"] == i, "2nd Sitting Mark"] = filtered.iloc[0, 20]
    Dataset_df.loc[Dataset_df["ID"] == i, "2nd Sitting Result"] = filtered.iloc[0, 21]
    Dataset_df.loc[Dataset_df["ID"] == i, "Attended Count"] = filtered.iloc[0, 22]
    Dataset_df.loc[Dataset_df["ID"] == i, "Not Attended"] = filtered.iloc[0, 23]
    Dataset_df.loc[Dataset_df["ID"] == i, "Moodle Count"] = filtered.iloc[0, 24]
    Dataset_df.loc[Dataset_df["ID"] == i, "Placement"] = filtered.iloc[0, 42]
    Dataset_df.loc[Dataset_df["ID"] == i, "HESA Qual. On Entry"] = filtered.iloc[0, 37]
    Dataset_df.loc[Dataset_df["ID"] == i, "HESA Short Code"] = filtered.iloc[0, 38]
  else:
    Dataset_df.loc[Dataset_df["ID"] == i, "Pass/Fail Module?"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "Mark"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "Result"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "Ass Grade"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "2nd Sitting Mark"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "2nd Sitting Result"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "Attended Count"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "Not Attended"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "Moodle Count"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "Placement"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "HESA Qual. On Entry"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "HESA Short Code"] = "N/A"

In [ ]:
second_df = xls.parse("2018 Second Sitting")

In [ ]:
Dataset_df["Sitting"] = ''

for i in Dataset_df["ID"].unique():
  filtered = second_df[second_df["ID"] == i]
  if len(filtered) > 0:
    Dataset_df.loc[Dataset_df["ID"] == i, "Sitting"] = filtered.iloc[0, 15]
  else:
    Dataset_df.loc[Dataset_df["ID"] == i, "Sitting"] = "N/A"

In [ ]:
Dataset_df["Start Date"] = ''
Dataset_df["End Date"] = ''

for i in Dataset_df["ID"].unique():
  filtered = withdraw_df[withdraw_df["ID"] == i]
  if len(filtered) > 0:
    Dataset_df.loc[Dataset_df["ID"] == i, "Start Date"] = filtered.iloc[0, 7]
    Dataset_df.loc[Dataset_df["ID"] == i, "End Date"] = filtered.iloc[0, 8]
  else:
    Dataset_df.loc[Dataset_df["ID"] == i, "Start Date"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "End Date"] = "N/A"

In [ ]:
Dataset_df["Course Year"] = ''
Dataset_df["Fee Cat"] = ''
Dataset_df["Nationality"] = ''
Dataset_df["Expec End Date (Course)"] = ''

for i in Dataset_df["ID"].unique():
  filtered = Foundation_df[Foundation_df["ID"] == i]
  if len(filtered) > 0:
    Dataset_df.loc[Dataset_df["ID"] == i, "Course Year"] = filtered.iloc[0, 6]
    Dataset_df.loc[Dataset_df["ID"] == i, "Fee Cat"] = filtered.iloc[0, 9]
    Dataset_df.loc[Dataset_df["ID"] == i, "Nationality"] = filtered.iloc[0, 14]
    Dataset_df.loc[Dataset_df["ID"] == i, "Expec End Date (Course)"] = filtered.iloc[0, 17]
  else:
    Dataset_df.loc[Dataset_df["ID"] == i, "Course Year"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "Fee Cat"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "Nationality"] = "N/A"
    Dataset_df.loc[Dataset_df["ID"] == i, "Expec End Date (Course)"] = "N/A"

In [ ]:
Dataset_df = pd.concat([Dataset_df, ModuleR_df, withdraw_df, Foundation_df, second_df], ignore_index=True)

In [ ]:
# Drop specific columns from the DataFrame
Dataset_df.drop(columns=['Cleaned Unit Code', 'WP Flag', 'Ac Year', 'Course','Description', 'Instance', 'Mode', 'Exit Date', 'HESA No. Of Years', 'Academic Year', 'Status','Course Stage','Student Stage','Cat','Start date','Ethnic Group Description', 'Hesa Location Of Study','Hesa Exchange',
                         'Start Date', 'End Date', 'Course Year', 'Fee Cat', 'Nationality', 'Expec End Date (Course)', 'Tariff on Entry',
                          'Ass Detail - Description', 'Ass Type', 'Pass/Fail Artefact?', 'Withdrawal Date'], inplace=True)

In [ ]:
Dataset_df['Moodle Count'] = pd.to_numeric(Dataset_df['Moodle Count'], errors='coerce')
Dataset_df['Attended Count'] = pd.to_numeric(Dataset_df['Attended Count'], errors='coerce')
Dataset_df['Not Attended'] = pd.to_numeric(Dataset_df['Not Attended'], errors='coerce')
Dataset_df['Mark'] = pd.to_numeric(Dataset_df['Mark'], errors='coerce')
column1_median = Dataset_df['Unit Average'].median()
column2_median = Dataset_df['Ass Res'].median()
column3_median = Dataset_df['Moodle Count'].median()
column4_median = Dataset_df['Attended Count'].median()
column5_median = Dataset_df['Not Attended'].median()
column6_median = Dataset_df['Mark'].median()


Dataset_df['Unit Average'].fillna(column1_median, inplace=True)
Dataset_df['Ass Res'].fillna(column2_median, inplace=True)
Dataset_df['Moodle Count'].fillna(column3_median, inplace=True)
Dataset_df['Attended Count'].fillna(column4_median, inplace=True)
Dataset_df['Not Attended'].fillna(column5_median, inplace=True)
Dataset_df['Mark'].fillna(column6_median, inplace=True)

In [ ]:
Change_NA_values = ['Withdrawal Reason', 'Sitting', 'Final Award', 'Unit Enrolment Status', 'Placement','Unit Result', 'Unit Grade', 'Assessment']
Dataset_df[Change_NA_values] = Dataset_df[Change_NA_values].fillna('Not applicable')

In [ ]:
Change_NA_values = ['Placement']

for column in Change_NA_values:
    Dataset_df[column] = Dataset_df[column].replace('N/A', 'Not applicable')

In [ ]:
change_value = ['2nd Sitting Mark', '2nd Sitting Result']

for column in change_value:
  Dataset_df[column] =  Dataset_df[column].replace('N/A', -1)

In [ ]:
Dataset_df.replace("N/A", "Not Applicable", inplace=True)
missing_values = Dataset_df.isna().sum()
print("Count of missing values (including 'N/A'):")
print(missing_values)

In [ ]:
Dataset_df["Distance from PO1"].replace("Missing", -1.0, inplace=True)

In [ ]:
Dataset_df["Pass/Fail Module?"].replace("Not Applicable", -1, inplace=True)

In [ ]:
Dataset_df["Placement"].replace("Not applicable", -1, inplace=True)

In [ ]:
from sklearn.preprocessing import LabelEncoder
Dataset_df = Dataset_df.astype(str)
backup = Dataset_df.copy()
categorical_columns = ['Dept', 'Course Name', 'Course Code', 'Course Instance', 'Grade', 'Student Status', 'Withdrawal Reason', 'Unit Code', 'Unit Name', 'Unit Reg Status', 'Unit Enrolment Status', 'Attend Group', 'Unit Result', 'Unit Grade', 'Assessment', 'Final Award', 'Gender', 'BME', 'Ethnicity Detail', 'Disability', 'Disability Detail', 'Mature', 'Term Postcode', 'Pass/Fail Module?', 'Result', 'Ass Grade', '2nd Sitting Result', '2nd Sitting Mark', 'HESA Qual. On Entry', 'HESA Short Code','Sitting']
label_encoder = LabelEncoder()
encoders_1 = {}

for column in categorical_columns:
    encoder = LabelEncoder()
    Dataset_df[column] = encoder.fit_transform(
        Dataset_df[column].astype(str)
    )
    encoders_1[column] = encoder

In [ ]:
import pandas as pd
from sklearn.feature_selection import SelectKBest, chi2

In [ ]:
 #1. Students who did a placement year and their outcomes
Dataset_df['Placement_Year'] = Dataset_df['Placement'].apply(lambda x: 1 if x == 'Yes' else 0)# placement

In [ ]:
#2. Students' gender, ethnicity, age group, and disability status
Dataset_df = pd.get_dummies(Dataset_df, columns=['Gender', 'Ethnicity Detail', 'Disability']) ## leave this stuff

In [ ]:
# 3. Academic features
Dataset_df['Unit_Grade_Mean'] = Dataset_df.groupby('ID')['Unit Grade'].transform('mean')
Dataset_df['Unit_Grade_Std'] = Dataset_df.groupby('ID')['Unit Grade'].transform('std')

In [ ]:
# 4. Performance trends
Dataset_df['Average_Grade'] = Dataset_df.groupby('ID')['Grade'].transform('mean')

In [ ]:
Dataset_df['Total_Assessment'] = Dataset_df['Assessment']
Dataset_df['Engagement_Ratio'] = Dataset_df['Assessment'] / Dataset_df['Total_Assessment']

In [ ]:
Dataset_df = pd.get_dummies(Dataset_df, columns=['Withdrawal Reason', 'Sitting']) ## leave this stuff

In [ ]:
# Dataset_df['Assessment_Ratio'] = Dataset_df['Ass Grade'] / Dataset_df['Assessment'] #theres an issue

In [ ]:
Dataset_df["Unit_Grade_Std"].fillna(Dataset_df["Unit_Grade_Std"].mean(), inplace=True)
Dataset_df["Engagement_Ratio"].fillna(Dataset_df["Engagement_Ratio"].mean(), inplace=True)

In [ ]:
from sklearn.preprocessing import MinMaxScaler
numerical_columns = ['Unit Average', '1st Sitting Unit Avg', '2nd Sitting Unit Avg', 'Mark', 'Attended Count', 'Not Attended', 'Moodle Count','Ass Res', 'Credits', 'Distance from PO1', 'Foundation', 'ICP','Pass/Fail Module?','Placement',
                     "Unit_Grade_Mean", "Unit_Grade_Std", "Average_Grade", "Total_Assessment", "Engagement_Ratio"]

scaler = MinMaxScaler()
Dataset_df[numerical_columns] = scaler.fit_transform(Dataset_df[numerical_columns])

In [ ]:
from sklearn.preprocessing import LabelEncoder
Dataset_df = Dataset_df.astype(str)
# categorical_columns = ['Dept', 'Course Name', 'Course Code', 'Course Instance', 'Grade', 'Student Status', 'Withdrawal Reason', 'Unit Code', 'Unit Name', 'Unit Reg Status', 'Unit Enrolment Status', 'Attend Group', 'Unit Result', 'Unit Grade', 'Assessment', 'Final Award', 'Gender', 'BME', 'Ethnicity Detail', 'Disability', 'Disability Detail', 'Mature', 'Term Postcode', 'Pass/Fail Module?', 'Result', 'Ass Grade', '2nd Sitting Result', '2nd Sitting Mark', 'HESA Qual. On Entry', 'HESA Short Code','Sitting']
categorical_columns = [i for i in Dataset_df.columns if i not in numerical_columns]
encoders = dict()
# Encode each categorical column using its own fitted encoder.
encoders = {}

for column in categorical_columns:
    encoder = LabelEncoder()
    Dataset_df[column] = encoder.fit_transform(
        Dataset_df[column].astype(str)
    )
    encoders[column] = encoder

In [ ]:
from sklearn.ensemble import RandomForestClassifier


In [ ]:
x = Dataset_df.drop(["Final Award"], axis=1)
y = Dataset_df.loc[:, "Grade"]

In [ ]:
x.head(10)

In [ ]:
tree = RandomForestClassifier()
tree.fit(x.values, y.values)

In [ ]:
importances = tree.feature_importances_
std = np.std([tree.feature_importances_ for tree in tree.estimators_], axis=0)

## 4. Exploratory Data Analysis and Feature Engineering

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt


forest_importances = pd.Series(importances, index=x.columns)

fig, ax = plt.subplots(figsize=(30, 10))
forest_importances.plot.bar(yerr=std, ax=ax)
ax.set_title("Feature importances using MDI")
ax.set_ylabel("Mean decrease in impurity")
fig.tight_layout()

In [ ]:
import tensorflow as tf
from tensorflow.keras.layers import Input, Embedding, Dense, Dropout
from tensorflow.keras.models import Model
from tensorflow import keras
from keras.utils import to_categorical

## 5. Feature Selection and Leakage Controls

In [ ]:
selected_features = [
    'Dept',
    'Course Name',
    'Course Instance',
    'Result',
    'Unit Code',
    'Unit Name',
    'Unit Enrolment Status',
    'Attend Group',
    'Unit Result',
    'Assessment',
    'Disability Detail',
    'Mature',
    'Term Postcode',
    'Grade',
    'Ass Grade',
    '2nd Sitting Result',
    'HESA Qual. On Entry',
    'HESA Short Code',
    'Gender_0',
    'Gender_1',
    'Ethnicity Detail_0',
    'Ethnicity Detail_1',
    'Ethnicity Detail_2',
    'Ethnicity Detail_3',
    'Ethnicity Detail_4',
    'Ethnicity Detail_5',
    'Ethnicity Detail_6',
    'Ethnicity Detail_7',
    'Disability_0',
    'Disability_1',
    'Unit Grade',
    '2nd Sitting Mark',
    'Sitting_0',
    'Sitting_1',
    'Sitting_2',
    'BME',

]
#Unit Reg Status

## 6. Multi-output Neural Network

In [ ]:
def create_multihead_attention_model(input_dim):
    inputs = Input(shape=(input_dim,))

    # embedding_layer = Embedding(input_dim=input_dim, output_dim=64)(inputs)
    # attention_output = MultiHeadAttention(num_heads=num_heads, key_dim=64)(embedding_layer, embedding_layer)
    layer1 = Dense(128, activation="relu")(inputs)
    layer2 = Dense(256, activation="relu")(layer1)
    layer3 = Dense(128, activation="relu")(layer2)
    # pooled_attention_output = GlobalAveragePooling1D()(attention_output)
    # concatenated = Concatenate(axis=-1)([inputs, pooled_attention_output])

    dense1 = Dense(64, activation='relu')(layer3)
    dropout = Dropout(0.5)(dense1)
    output_final_award = Dense(Dataset_df["Final Award"].nunique(), activation='softmax', name='final_award_output')(dropout)
    output_student_status = Dense(Dataset_df["Unit Reg Status"].nunique(), activation='softmax', name='student_status_output')(dropout)

    model = Model(inputs=inputs, outputs=[output_final_award, output_student_status])
    return model

input_dim = len(selected_features)

model = create_multihead_attention_model(input_dim)
print(f"Number of modelling features: {input_dim}")
model.compile(optimizer='adam',
              loss={'final_award_output': "sparse_categorical_crossentropy",
                    'student_status_output': "sparse_categorical_crossentropy"},
              metrics=['accuracy'])


## 7. Student-Level Grouped Data Split

In [ ]:
# Student-level grouped splitting prevents records belonging to the same
# student from appearing in both training and evaluation datasets.

from sklearn.model_selection import GroupShuffleSplit
import numpy as np
import matplotlib.pyplot as plt
from tensorflow.keras.callbacks import EarlyStopping

X = Dataset_df[selected_features].values
y = Dataset_df[["Final Award", "Unit Reg Status"]].values
groups = student_groups.loc[Dataset_df.index].values

# Hold out 20% of students for the final test set.
test_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.20,
    random_state=42,
)

train_val_idx, test_idx = next(test_split.split(X, y, groups=groups))

X_train_val = X[train_val_idx]
y_train_val = y[train_val_idx]
groups_train_val = groups[train_val_idx]

X_test = X[test_idx]
y_test = y[test_idx]

# Use 25% of the remaining students for validation:
# 60% training, 20% validation and 20% testing overall.
validation_split = GroupShuffleSplit(
    n_splits=1,
    test_size=0.25,
    random_state=42,
)

train_idx, val_idx = next(
    validation_split.split(
        X_train_val,
        y_train_val,
        groups=groups_train_val,
    )
)

X_train = X_train_val[train_idx]
y_train = y_train_val[train_idx]
X_val = X_train_val[val_idx]
y_val = y_train_val[val_idx]

print(f"Training records: {len(X_train):,}")
print(f"Validation records: {len(X_val):,}")
print(f"Testing records: {len(X_test):,}")

print(
    "Student overlap:",
    len(set(groups[train_val_idx]).intersection(set(groups[test_idx]))),
)


early_stopping = EarlyStopping(monitor='val_loss', patience=3, restore_best_weights=True)
# Train the model
history = model.fit(
    X_train,
    {'final_award_output': y_train[:, 0],
     'student_status_output': y_train[:, 1]},
    validation_data=(
        X_val,
        {'final_award_output': y_val[:, 0],
         'student_status_output': y_val[:, 1]}),
    epochs=50,
    batch_size=32,
    callbacks=[early_stopping]
)

# Evaluate the model
results = model.evaluate(
    X_test,
    {'final_award_output': y_test[:, 0],
     'student_status_output': y_test[:, 1]},
    verbose=0
)

print("Loss:", results[0])
print("Final Award Accuracy:", results[1])
print("Student Status Accuracy:", results[2])


plt.plot(history.history['final_award_output_accuracy'])
plt.plot(history.history['student_status_output_accuracy'])
plt.title('Model accuracy')
plt.ylabel('Accuracy')
plt.xlabel('Epoch')
plt.legend(['Final Award Accuracy', 'Student Status Accuracy'], loc='upper left')
plt.show()


plt.plot(history.history['loss'])
plt.plot(history.history['val_loss'])
plt.title('Model loss')
plt.ylabel('Loss')
plt.xlabel('Epoch')
plt.legend(['Training loss', 'Validation loss'], loc='upper left')
plt.show()


## 8. Classification Report

In [ ]:
from sklearn.metrics import classification_report
predicted_final_award, predicted_student_status = model.predict(X_test)

threshold = 0.5
predicted_final_award_class = np.argmax(predicted_final_award, axis=1)
predicted_student_status_class = np.argmax(predicted_student_status, axis=1)

y_true_final_award = y_test[:, 0]
y_true_student_status = y_test[:, 1]

print("Final Award Classification Report:")
print(classification_report(y_true_final_award, predicted_final_award_class))
print("Student Status Classification Report:")
print(classification_report(y_true_student_status, predicted_student_status_class))

In [ ]:
backup["Unit Reg Status"].unique()

In [ ]:
backup["Final Award"].unique()

In [ ]:
encoders["Unit Reg Status"].inverse_transform(Dataset_df["Unit Reg Status"].values)

##9. Random Forest Model

In [ ]:
from sklearn.ensemble import RandomForestClassifier

# Create the Random Forest classifier
rf_model = RandomForestClassifier(n_estimators=100, random_state=42)

# Train the model
rf_model.fit(X_train, y_train[:, 0])  # Assuming 'Final Award' is the first column in y_train

# Evaluate the model
rf_accuracy = rf_model.score(X_test, y_test[:, 0])  # Assuming 'Final Award' is the first column in y_test

print("Random Forest Final Award Accuracy:", rf_accuracy)


In [ ]:
from sklearn.multioutput import MultiOutputClassifier
from sklearn.ensemble import RandomForestClassifier

# Create the Random Forest classifier
rf_classifier = RandomForestClassifier(n_estimators=100, random_state=42)

# Wrap the classifier in MultiOutputClassifier
multi_target_rf = MultiOutputClassifier(rf_classifier, n_jobs=-1)

# Train the model
multi_target_rf.fit(X_train, y_train)

# Evaluate the model
accuracy = multi_target_rf.score(X_test, y_test)

print("Random Forest Accuracy:", accuracy)

## 10. Confusion Matrix

In [ ]:
from sklearn.metrics import confusion_matrix
import matplotlib.pyplot as plt
import numpy as np

def plot_confusion_matrix(y_true, y_pred, classes,
                          normalize=False,
                          title=None,
                          cmap=plt.cm.Blues):
    """
    This function prints and plots the confusion matrix.
    Normalization can be applied by setting `normalize=True`.
    """
    if not title:
        if normalize:
            title = 'Normalized confusion matrix'
        else:
            title = 'Confusion matrix, without normalization'


    cm = confusion_matrix(y_true, y_pred)
    if normalize:
        cm = cm.astype('float') / cm.sum(axis=1)[:, np.newaxis]
        print("Normalized confusion matrix")
    else:
        print('Confusion matrix, without normalization')

    fig, ax = plt.subplots()
    im = ax.imshow(cm, interpolation='nearest', cmap=cmap)
    ax.figure.colorbar(im, ax=ax)

    ax.set(xticks=np.arange(cm.shape[1]),
           yticks=np.arange(cm.shape[0]),
           xticklabels=classes, yticklabels=classes,
           title=title,
           ylabel='True label',
           xlabel='Predicted label')


    plt.setp(ax.get_xticklabels(), rotation=45, ha="right",
             rotation_mode="anchor")


    fmt = '.2f' if normalize else 'd'
    thresh = cm.max() / 2.
    for i in range(cm.shape[0]):
        for j in range(cm.shape[1]):
            ax.text(j, i, format(cm[i, j], fmt),
                    ha="center", va="center",
                    color="white" if cm[i, j] > thresh else "black")
    fig.tight_layout()
    return ax



plot_confusion_matrix(y_test[:, 0], y_test[:, 0], classes=np.unique(y_test[:, 0]),
                      title='Confusion matrix - Final Award')


plot_confusion_matrix(y_test[:, 1], y_test[:, 1], classes=np.unique(y_test[:, 1]), normalize=True,
                      title='Normalized confusion matrix - Student Status')

plt.show()


## 11. XGBoost Model




In [ ]:
import xgboost as xgb
from sklearn.multioutput import MultiOutputClassifier

xgb_model = MultiOutputClassifier(xgb.XGBClassifier(objective='binary:logistic'))


xgb_model.fit(X_train, y_train)


accuracy = xgb_model.score(X_test, y_test)

print("Accuracy:", accuracy)


## 12. Preliminary Results

The original academic implementation reported the following test-set results:

| Model | Prediction task | Reported accuracy |
|---|---|---:|
| Multi-output neural network | Final academic award | 65.0% |
| Random Forest | Final academic award | 88.9% |
| Multi-output neural network | Student status | 99.6% |
| XGBoost | Multi-output prediction | 90.2% |

These figures were produced by the original implementation and are retained as historical project results. They should not be interpreted as validated deployment performance.

The revised public code introduces student-level grouped splitting and removes obvious target-leakage features. Consequently, rerunning it on authorised data may produce lower—but more reliable—performance estimates.

### Key Interpretation

- Random Forest produced the strongest reported final-award accuracy.
- The extremely high student-status accuracy requires caution because of class imbalance and possible leakage in the original evaluation.
- Accuracy alone is insufficient for evaluating withdrawal risk; recall, precision, macro F1 and balanced accuracy should also be assessed.
- Model outputs should be used to identify students who may benefit from support, never to make punitive or fully automated decisions.